#Smart Medical Question Answering System using LoRA
Problem Statement:

A hospital wants an AI assistant capable of answering medical questions. Training an entire Large Language Model (LLM) is expensive and requires significant computational resources.

Your task is to use LoRA (Low-Rank Adaptation) to efficiently adapt a pretrained LLM for medical question answering while training only a small subset of parameters.

#Task 1: Install Required Libraries
Objective:
Prepare the environment for LoRA-based fine-tuning.

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
import transformers
import datasets
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 5.11.0
Datasets: 4.0.0
PEFT: 0.19.1
Accelerate: 1.13.0
BitsAndBytes: 0.49.2


#Task 2: Load a Pretrained LLM
Objective:
Load a pretrained TinyLlama model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

print(type(base_model))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


#Task 3: Analyze Model Parameters
Objective:
Calculate total parameters.

In [ ]:
total_params = sum(
    p.numel()
    for p in base_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in base_model.parameters()
    if p.requires_grad
)

trainable_pct = 100 * trainable_params / total_params

print("Task 3 Results")
print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Trainable Percentage  : {trainable_pct:.4f}%")

Task 3 Results
Total Parameters      : 1,100,048,384
Trainable Parameters  : 1,100,048,384
Trainable Percentage  : 100.0000%


#Task 4: Apply LoRA
Objective:

Attach LoRA adapters to the pretrained model.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(
    base_model,
    lora_config
)

print(type(lora_model))

<class 'peft.peft_model.PeftModelForCausalLM'>


#Task 5: Verify Trainable Parameters
Objective:
Compare trainable parameters before and after LoRA.

In [ ]:
before_total = total_params
before_trainable = trainable_params
before_pct = trainable_pct

after_total = sum(
    p.numel()
    for p in lora_model.parameters()
)

after_trainable = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

after_pct = 100 * after_trainable / after_total

print("AFTER LoRA")
print(f"Total Parameters      : {after_total:,}")
print(f"Trainable Parameters  : {after_trainable:,}")
print(f"Trainable Percentage  : {after_pct:.4f}%")

AFTER LoRA
Total Parameters      : 1,101,174,784
Trainable Parameters  : 1,126,400
Trainable Percentage  : 0.1023%


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Total Parameters",
        "Trainable Parameters",
        "Trainable Percentage"
    ],
    "Before LoRA": [
        f"{before_total:,}",
        f"{before_trainable:,}",
        f"{before_pct:.4f}%"
    ],
    "After LoRA": [
        f"{after_total:,}",
        f"{after_trainable:,}",
        f"{after_pct:.4f}%"
    ]
})

comparison

,Metric,Before LoRA,After LoRA
0,Total Parameters,"1,100,048,384","1,101,174,784"
1,Trainable Parameters,"1,100,048,384","1,126,400"
2,Trainable Percentage,100.0000%,0.1023%


#Task 6: Create Medical QA Dataset
Objective:
Prepare a small medical dataset.


In [ ]:
from datasets import Dataset

data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 3
})


In [ ]:
print(dataset[0])

{'question': 'What is diabetes?', 'answer': 'Diabetes is a chronic disease where the body cannot regulate blood sugar properly.'}


In [ ]:
def format_examples(example):
    return {
        "text": f"Question: {example['question']}\nAnswer: {example['answer']}"
    }

formatted_dataset = dataset.map(format_examples)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Question: What is diabetes?
Answer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.


In [ ]:
for item in formatted_dataset:
    print(item["text"])
    print("-" * 50)

Question: What is diabetes?
Answer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.
--------------------------------------------------
Question: How to treat a common cold?
Answer: Rest, hydration, and over-the-counter medications can help manage symptoms.
--------------------------------------------------
Question: What are the symptoms of hypertension?
Answer: Symptoms include headaches, shortness of breath, and nosebleeds.
--------------------------------------------------


#Task 7: Tokenization
Objective:
Convert text into tokens.

Deliverables

Display:
Input IDs ,
Attention Masks

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
print("Input IDs:")
print(tokenized_dataset[0]["input_ids"])

Input IDs:
[1, 894, 29901, 1724, 338, 652, 370, 10778, 29973, 13, 22550, 29901, 4671, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [ ]:
print("Attention Mask:")
print(tokenized_dataset[0]["attention_mask"])

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
sample = tokenized_dataset[0]

print("Question + Answer:")
print(formatted_dataset[0]["text"])

print("\nInput IDs:")
print(sample["input_ids"])

print("\nAttention Mask:")
print(sample["attention_mask"])

Question + Answer:
Question: What is diabetes?
Answer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.

Input IDs:
[1, 894, 29901, 1724, 338, 652, 370, 10778, 29973, 13, 22550, 29901, 4671, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
decoded_text = tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=True
)

print(decoded_text)

Question: What is diabetes?
Answer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.


In [ ]:
print(tokenized_dataset[0]["input_ids"])
print(tokenized_dataset[0]["attention_mask"])

[1, 894, 29901, 1724, 338, 652, 370, 10778, 29973, 13, 22550, 29901, 4671, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


#Task 8: Fine-Tune Using LoRA
Objective:
Train only LoRA layers.

In [ ]:
tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": x["input_ids"]}
)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./medical_lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Step,Training Loss
1,1.785287
2,1.815033
3,1.916721
4,1.912337
5,1.765398
6,1.789640
7,1.895994
8,1.787169
9,1.753177


TrainOutput(global_step=9, training_loss=1.8245285749435425, metrics={'train_runtime': 3.6704, 'train_samples_per_second': 2.452, 'train_steps_per_second': 2.452, 'total_flos': 7158335275008.0, 'train_loss': 1.8245285749435425, 'epoch': 3.0})

In [ ]:
lora_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


#Task 9: Save LoRA Adapter
Objective:
Save only adapter weights.

In [ ]:
lora_model.save_pretrained("./medical_lora_adapter")

In [ ]:
tokenizer.save_pretrained("./medical_lora_adapter")

('./medical_lora_adapter/tokenizer_config.json',
 './medical_lora_adapter/chat_template.jinja',
 './medical_lora_adapter/tokenizer.json')

In [ ]:
import os

print(os.listdir("./medical_lora_adapter"))

['chat_template.jinja', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'README.md']


In [ ]:
import os

adapter_file = "./medical_lora_adapter/adapter_model.safetensors"

size_mb = os.path.getsize(adapter_file) / (1024 * 1024)

print(f"Adapter Size: {size_mb:.2f} MB")

Adapter Size: 4.31 MB


In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

model = PeftModel.from_pretrained(
    base_model,
    "./medical_lora_adapter"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

#Task 10: Medical Question Answering
Objective:
Generate answers using the fine-tuned model.

In [ ]:
question = "What is diabetes?"

prompt = f"Question: {question}\nAnswer:"

In [ ]:
import torch

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to(base_model.device)
    for k, v in inputs.items()
}

outputs = base_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

before_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("BEFORE LoRA")
print(before_response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEFORE LoRA
Question: What is diabetes?
Answer: Diabetes is a chronic disease that occurs when the body does not produce or use insulin properly. Insulin is a hormone that helps the body use glucose (sugar) from the bloodstream. When


In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to(lora_model.device)
    for k, v in inputs.items()
}

outputs = lora_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

after_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("AFTER LoRA")
print(after_response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER LoRA
Question: What is diabetes?
Answer: Diabetes is a chronic disease that occurs when the body does not produce or use insulin properly. Insulin is a hormone that helps the body use glucose (sugar) from the bloodstream. When


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Question": [question],
    "Before LoRA": [before_response],
    "After LoRA": [after_response]
})

comparison

,Question,Before LoRA,After LoRA
0,What is diabetes?,Question: What is diabetes?\nAnswer: Diabetes ...,Question: What is diabetes?\nAnswer: Diabetes ...


In [ ]:
questions = [
    "What is diabetes?",
    "How to treat a common cold?",
    "What are the symptoms of hypertension?"
    "What is hyperhydrosis?"
]

for question in questions:

    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(lora_model.device)
        for k, v in inputs.items()
    }

    outputs = lora_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("="*60)
    print(answer)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is diabetes?
Answer: Diabetes is a chronic disease that occurs when the body does not produce or use insulin properly. Insulin is a hormone that helps the body use glucose (sugar) from the bloodstream. When the body does not produce enough insulin or


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How to treat a common cold?
Answer: A common cold is a viral infection that causes sore throat, cough, runny nose, and fever. The best treatment for a common cold is rest, drinking plenty of fluids, and taking over-the-counter medications such as cough syrup,
Question: What are the symptoms of hypertension?What is hyperhydrosis?
Answer: Hypertension is a medical condition in which the blood pressure is higher than normal. The symptoms of hypertension include headaches, dizziness, fatigue, and shortness of breath. Hyperhydrosis is a condition in which excessive sweating occurs. The symptoms
